In [1]:
# Load configuration from config.py
from config import CONFIG, print_config

# Print configuration summary
print_config()

Configuration loaded:
  data_dir: ../../data
  output_file: ./submission.csv
  embedding_model: intfloat/e5-small-v2
  reranker_model: cross-encoder/ms-marco-MiniLM-L-6-v2
  top_k_retrieval: 50
  top_k_final: 10
  embed_batch_size: 8
  rerank_batch_size: 8
  max_length: 4096
  datasets: 7 datasets


# FinanceRAG - Full Retrieval Pipeline

**Objective:** Retrieve top 10 most relevant documents for each query across 7 financial datasets.

**Strategy:** 
1. Process each dataset separately (Divide & Conquer)
2. Use Bi-Encoder (BGE-M3) for fast retrieval (Top-50)
3. Use Cross-Encoder (BGE-Reranker) for precise reranking (Top-10)
4. Combine all results into submission file

**No Generation needed** - Pure Retrieval Task!

In [2]:
# Core Libraries
import os
import pandas as pd
import json
import numpy as np
from tqdm.auto import tqdm
from typing import List, Dict, Tuple

# Embedding & Retrieval
from sentence_transformers import SentenceTransformer
import faiss

# Reranking
from sentence_transformers import CrossEncoder

# Utils
import warnings
warnings.filterwarnings('ignore')

import logging
logging.disable(logging.CRITICAL)

In [3]:
# Check GPU availability and set device
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA version:", torch.version.cuda)
    print("GPU:", torch.cuda.get_device_name(0))    
    device = 'cuda'
else:
    print("Running on CPU")
    device = 'cpu'

print(f"\nUsing device: {device}")

CUDA available: True
CUDA version: 12.1
GPU: NVIDIA GeForce RTX 3050 Laptop GPU

Using device: cuda


## Helper Functions

In [4]:
# Import shared utilities
import sys
sys.path.insert(0, '..')
from utils import (
    DATASETS, 
    QRELS_MAPPING, 
    compute_ndcg_batch as compute_ndcg,
    evaluate_results_df as evaluate_results
)

# E5 model prefixes (required for E5 models!)
E5_QUERY_PREFIX = "query: "
E5_PASSAGE_PREFIX = "passage: "


def load_jsonl_data(dataset_name: str, data_dir: str) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Load corpus, queries, and qrels for a given dataset.
    
    Note: In this project, corpus files are in subfolders:
    e.g., data/financebench_corpus.jsonl/corpus.jsonl
    """
    # Construct paths (corpus is in subfolder)
    corpus_path = os.path.join(data_dir, f"{dataset_name}_corpus.jsonl", "corpus.jsonl")
    queries_path = os.path.join(data_dir, f"{dataset_name}_queries.jsonl", "queries.jsonl")
    
    # Qrels file mapping
    qrels_mapping = {
        'convfinqa': 'ConvFinQA_qrels.tsv',
        'financebench': 'FinanceBench_qrels.tsv',
        'finder': 'FinDER_qrels.tsv',
        'finqa': 'FinQA_qrels.tsv',
        'finqabench': 'FinQABench_qrels.tsv',
        'multiheirtt': 'MultiHeirtt_qrels.tsv',
        'tatqa': 'TATQA_qrels.tsv'
    }
    qrels_path = os.path.join(data_dir, qrels_mapping.get(dataset_name, f"{dataset_name}_qrels.tsv"))
    
    # Check if files exist
    if not os.path.exists(corpus_path):
        raise FileNotFoundError(f"Corpus not found: {corpus_path}")
    if not os.path.exists(queries_path):
        raise FileNotFoundError(f"Queries not found: {queries_path}")
    
    # Load data
    corpus_df = pd.read_json(corpus_path, lines=True)
    queries_df = pd.read_json(queries_path, lines=True)
    
    # Load qrels if exists
    qrels_df = None
    if os.path.exists(qrels_path):
        qrels_df = pd.read_csv(qrels_path, sep='\t')
    
    print(f"  Loaded {len(corpus_df)} corpus documents, {len(queries_df)} queries", end="")
    if qrels_df is not None:
        print(f", {len(qrels_df)} qrels")
    else:
        print()
    
    return corpus_df, queries_df, qrels_df


def prepare_texts(df: pd.DataFrame, combine_title: bool = True, is_query: bool = False) -> List[str]:
    """
    Prepare text from dataframe for embedding.
    
    For E5 models, we need to add prefixes:
    - Queries: "query: <text>"
    - Documents: "passage: <text>"
    
    For financial documents, combining title + text is crucial:
    - Title often contains company name, year, report type
    - Helps disambiguate between similar documents
    """
    # Determine prefix for E5 models
    prefix = E5_QUERY_PREFIX if is_query else E5_PASSAGE_PREFIX
    
    if combine_title and 'title' in df.columns:
        # Combine title and text with proper formatting
        texts = []
        for _, row in df.iterrows():
            title = str(row.get('title', '')).strip()
            text = str(row.get('text', '')).strip()
            if title and text:
                combined = f"{title}. {text}" 
            elif title:
                combined = title
            else:
                combined = text
            # Add E5 prefix
            texts.append(f"{prefix}{combined}")
        return texts
    else:
        # Add E5 prefix to each text
        return [f"{prefix}{str(t)}" for t in df['text'].astype(str).tolist()]


def build_faiss_index(embeddings: np.ndarray, use_gpu: bool = False) -> faiss.Index:
    """
    Build FAISS index for fast similarity search.
    
    Using IndexFlatIP (Inner Product) because BGE models output normalized vectors.
    """
    dimension = embeddings.shape[1]
    
    # Create index
    index = faiss.IndexFlatIP(dimension)
    
    # Move to GPU if available and requested
    if use_gpu and faiss.get_num_gpus() > 0:
        print("  Using GPU for FAISS")
        res = faiss.StandardGpuResources()
        index = faiss.index_cpu_to_gpu(res, 0, index)
    
    return index


print("✅ Helper functions defined (shared constants from utils.py)")
print(f"   E5 Query prefix: '{E5_QUERY_PREFIX}'")
print(f"   E5 Passage prefix: '{E5_PASSAGE_PREFIX}'")
print("✅ Evaluation functions imported: compute_ndcg, evaluate_results")

✅ Helper functions defined (shared constants from utils.py)
   E5 Query prefix: 'query: '
   E5 Passage prefix: 'passage: '
✅ Evaluation functions imported: compute_ndcg, evaluate_results


## Load Models (Once for All Datasets)

In [5]:
print("Loading models... This may take a few minutes on first run.")

# 1. Bi-Encoder for Retrieval (Fast, captures semantic similarity)
print(f"\n1. Loading embedding model: {CONFIG['embedding_model']}")
embed_model = SentenceTransformer(
    CONFIG['embedding_model'], 
    device=device,
    trust_remote_code=True  # Allow loading custom model architectures
)
print(f"   Model loaded on {device}")

# 2. Cross-Encoder for Reranking (Slower but more accurate)
print(f"\n2. Loading reranker model: {CONFIG['reranker_model']}")
reranker = CrossEncoder(
    CONFIG['reranker_model'], 
    device=device, 
    max_length=512
)
print(f"   Reranker loaded on {device}")

print("\n✅ All models loaded successfully!")

Loading models... This may take a few minutes on first run.

1. Loading embedding model: intfloat/e5-small-v2
   Model loaded on cuda

2. Loading reranker model: cross-encoder/ms-marco-MiniLM-L-6-v2
   Reranker loaded on cuda

✅ All models loaded successfully!


## Main Processing Function

In [6]:
def process_dataset(dataset_name: str, config: Dict) -> Tuple[pd.DataFrame, Dict]:
    """
    Complete pipeline for one dataset:
    1. Load data
    2. Embed corpus
    3. Build FAISS index
    4. Retrieve top-K candidates
    5. Rerank to top-10
    6. Evaluate (if qrels available)
    7. Return results
    """
    print(f"\n{'='*60}")
    print(f"Processing: {dataset_name.upper()}")
    print(f"{'='*60}")
    
    # --- STEP 1: Load Data (including qrels for evaluation) ---
    corpus_df, queries_df, qrels_df = load_jsonl_data(dataset_name, config['data_dir'])
    
    # Prepare texts for embedding (with E5 prefixes)
    corpus_texts = prepare_texts(corpus_df, combine_title=True, is_query=False)  # passage prefix
    corpus_ids = corpus_df['_id'].tolist()
    
    # Store raw corpus texts for reranking (without prefix)
    corpus_texts_raw = []
    for _, row in corpus_df.iterrows():
        title = str(row.get('title', '')).strip()
        text = str(row.get('text', '')).strip()
        if title and text:
            corpus_texts_raw.append(f"{title}. {text}")
        elif title:
            corpus_texts_raw.append(title)
        else:
            corpus_texts_raw.append(text)
    
    query_texts = prepare_texts(queries_df, combine_title=False, is_query=True)  # query prefix
    query_ids = queries_df['_id'].tolist()
    
    # Store raw query texts for reranking (without prefix)
    query_texts_raw = queries_df['text'].astype(str).tolist()
    
    # --- STEP 2: Embed Corpus ---
    print(f"\n📊 Embedding {len(corpus_texts)} documents...")
    max_length = config.get('max_length', 4096)
    corpus_embeddings = embed_model.encode(
        corpus_texts,
        batch_size=config['embed_batch_size'],
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True
    )
    
    # --- STEP 3: Build FAISS Index ---
    print(f"\n🔍 Building FAISS index...")
    dimension = corpus_embeddings.shape[1]
    index = faiss.IndexFlatIP(dimension)
    index.add(corpus_embeddings.astype('float32'))
    print(f"   Index built with {index.ntotal} vectors")
    
    # Free memory
    del corpus_embeddings
    if device == 'cuda':
        import torch
        torch.cuda.empty_cache()
    
    # --- STEP 4: Retrieve Top-K Candidates ---
    print(f"\n🎯 Retrieving top-{config['top_k_retrieval']} candidates for {len(query_texts)} queries...")
    query_embeddings = embed_model.encode(
        query_texts,
        batch_size=config['embed_batch_size'],
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True
    )
    
    # Search FAISS index
    distances, indices = index.search(
        query_embeddings.astype('float32'),
        config['top_k_retrieval']
    )
    
    # Free memory
    del query_embeddings
    if device == 'cuda':
        import torch
        torch.cuda.empty_cache()
    
    # --- STEP 5: Rerank to Top-10 ---
    print(f"\n⚡ Reranking to top-{config['top_k_final']}...")
    results = []
    rerank_batch_size = config.get('rerank_batch_size', 8)
    
    for i, query_id in enumerate(tqdm(query_ids, desc="Reranking")):
        query_text = query_texts_raw[i]  # Use raw text (no prefix) for reranking
        
        # Get candidates from retrieval step
        candidate_indices = indices[i]
        candidate_texts = [corpus_texts_raw[idx] for idx in candidate_indices]  # Use raw texts
        candidate_ids = [corpus_ids[idx] for idx in candidate_indices]
        
        # Truncate candidate texts to prevent OOM in reranker
        max_rerank_len = 512
        candidate_texts = [text[:max_rerank_len*4] for text in candidate_texts]
        
        # Create pairs for reranker
        pairs = [[query_text, doc_text] for doc_text in candidate_texts]
        
        # Get reranking scores
        scores = reranker.predict(pairs, show_progress_bar=False, batch_size=rerank_batch_size)
        
        # Sort by score and take top-10
        scored_candidates = list(zip(candidate_ids, scores))
        scored_candidates.sort(key=lambda x: x[1], reverse=True)
        top_10 = scored_candidates[:config['top_k_final']]
        
        # Store results
        for corpus_id, score in top_10:
            results.append({
                'query_id': query_id,
                'corpus_id': corpus_id,
                'score': float(score)
            })
    
    results_df = pd.DataFrame(results)
    print(f"\n✅ Completed {dataset_name}: {len(results_df)} results")
    
    # --- STEP 6: Evaluate (if qrels available) ---
    eval_metrics = {}
    if config.get('eval_on_qrels', True) and qrels_df is not None:
        print(f"\n📊 Evaluating on qrels...")
        eval_metrics = evaluate_results(results_df, qrels_df)
        print(f"   NDCG@10: {eval_metrics['NDCG@10']:.4f}")
        print(f"   Queries evaluated: {eval_metrics['num_qrels']}")
    
    # Clear GPU cache
    if device == 'cuda':
        import torch
        torch.cuda.empty_cache()
        print(f"   GPU cache cleared")
    
    return results_df, eval_metrics

## Run Pipeline for All Datasets

## Generate Submission File

In [7]:
all_results = []
all_eval = {}  # Store evaluation results per dataset
failed_datasets = []

print(f"Starting pipeline for {len(CONFIG['datasets'])} datasets...\n")

for dataset_name in CONFIG['datasets']:
    try:
        df_results, eval_metrics = process_dataset(dataset_name, CONFIG)
        all_results.append(df_results)
        if eval_metrics:
            all_eval[dataset_name] = eval_metrics
    except Exception as e:
        print(f"\n❌ Error processing {dataset_name}: {str(e)}")
        import traceback
        traceback.print_exc()
        failed_datasets.append(dataset_name)
        continue

print(f"\n{'='*60}")
print(f"Pipeline completed!")
print(f"  Successful: {len(all_results)}/{len(CONFIG['datasets'])} datasets")
if failed_datasets:
    print(f"  Failed: {', '.join(failed_datasets)}")
print(f"{'='*60}")

Starting pipeline for 7 datasets...


Processing: CONVFINQA
  Loaded 2066 corpus documents, 421 queries, 126 qrels

📊 Embedding 2066 documents...


Batches:   0%|          | 0/259 [00:00<?, ?it/s]


🔍 Building FAISS index...
   Index built with 2066 vectors

🎯 Retrieving top-50 candidates for 421 queries...


Batches:   0%|          | 0/53 [00:00<?, ?it/s]


⚡ Reranking to top-10...


...


🔍 Building FAISS index...
   Index built with 2756 vectors

🎯 Retrieving top-50 candidates for 1663 queries...


Batches:   0%|          | 0/208 [00:00<?, ?it/s]


⚡ Reranking to top-10...


Reranking:   0%|          | 0/1663 [00:00<?, ?it/s]


✅ Completed tatqa: 16630 results

📊 Evaluating on qrels...
   NDCG@10: 0.3762
   Queries evaluated: 498
   GPU cache cleared

Pipeline completed!
  Successful: 7/7 datasets


## 📊 Evaluation Summary

In [8]:
if all_eval:
    print("\n📊 Local Evaluation (30% qrels):")
    print("="*60)
    
    total_ndcg = 0
    total_queries = 0
    
    for ds, m in all_eval.items():
        print(f"\n{ds.upper()}: NDCG@10 = {m['NDCG@10']:.4f}")
        total_ndcg += m['NDCG@10'] * m['num_qrels']
        total_queries += m['num_qrels']
    
    if total_queries > 0:
        avg_ndcg = total_ndcg / total_queries
        print(f"\n{'='*60}")
        print(f"📈 AVERAGE NDCG@10: {avg_ndcg:.4f}")
        print(f"{'='*60}")
        
        baseline = 0.3280
        gain = avg_ndcg - baseline
        gain_pct = (gain / baseline) * 100
        
        print(f"\n🎯 vs Baseline:")
        print(f"   Baseline: {baseline:.4f}")
        print(f"   Improved: {avg_ndcg:.4f}")
        print(f"   Gain: +{gain:.4f} ({gain_pct:+.1f}%)")
        
        if avg_ndcg >= 0.58:
            print(f"\n🏆 Likely TOP 3!")
        elif avg_ndcg >= 0.50:
            print(f"\n✅ Good progress, tune more!")
        else:
            print(f"\n⚠️ Need more work")


📊 Local Evaluation (30% qrels):

CONVFINQA: NDCG@10 = 0.3983

FINANCEBENCH: NDCG@10 = 0.7094

FINDER: NDCG@10 = 0.3759

FINQA: NDCG@10 = 0.3317

FINQABENCH: NDCG@10 = 0.8755

MULTIHEIRTT: NDCG@10 = 0.1120

TATQA: NDCG@10 = 0.3762

📈 AVERAGE NDCG@10: 0.3335

🎯 vs Baseline:
   Baseline: 0.3280
   Improved: 0.3335
   Gain: +0.0055 (+1.7%)

⚠️ Need more work


In [ ]:
# Combine all results
if all_results:
    final_df = pd.concat(all_results, ignore_index=True)
    
    # Submission format: query_id, corpus_id (no score column)
    submission_df = final_df[['query_id', 'corpus_id']]
    
    # Save to CSV
    output_path = CONFIG['output_file']
    submission_df.to_csv(output_path, index=False)
    
    print(f"\n✅ Submission file saved: {output_path}")
    print(f"   Total entries: {len(submission_df)}")
    print(f"   Unique queries: {submission_df['query_id'].nunique()}")
    print(f"   Expected format: query_id, corpus_id")
    
    # Show sample
    print(f"\n📋 First 10 rows:")
    print(submission_df.head(10))
    
    # Validation checks
    print(f"\n🔍 Validation:")
    print(f"   - Each query should have 10 results: {submission_df.groupby('query_id').size().value_counts().to_dict()}")
    print(f"   - No null values: {submission_df.isnull().sum().sum() == 0}")
else:
    print("\n❌ No results to save. All datasets failed.")


✅ Submission file saved: ./submission.csv
   Total entries: 46710
   Unique queries: 4671
   Expected format: query_id, corpus_id

📋 First 10 rows:
    query_id  corpus_id
0  qd4982518  dd4bb5506
1  qd4982518  dd4bb016e
2  qd4982518  dd4b9f7f6
3  qd4982518  dd4bf5c14
4  qd4982518  dd4be45d6
5  qd4982518  dd4bf6f9c
6  qd4982518  dd4c4f7aa
7  qd4982518  dd4bf1060
8  qd4982518  dd4b87d18
9  qd4982518  dd4bc0884

🔍 Validation:
   - Each query should have 10 results: {10: 4671}
   - No null values: True


: 